In [1]:

# path = '/content/drive/MyDrive/SchwarMAX-MCMC/'
path = '/Users/hanyuan/Dropbox/python_script/SchwarMAX/'

import sys
sys.path.append(path)

from model import *
from likelihoods import *
from utils import *
from sample_from_density import sample_from_density_grid
from CylindricalSpline import get_phi_m, evaluate_phi_axisymmetric

import os
os.environ["JAX_ENABLE_X64"] = "True"

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.numpy.linalg as jnn
import pandas as pd
import numpy as np
import scipy as sp
import pickle

import emcee
import corner
import matplotlib.pyplot as plt

from constants import EPSILON

import time

def get_dict_data(path):

    with open(path + 'mock_Nbody_bar_XY_withRot.pkl', 'rb') as f:
        bin_dict = pickle.load(f)

    # voronoi binning mapping and data
    num_per_bin = jnp.array(bin_dict['num_per_bin'])
    total_bins = jnp.array(bin_dict['total_bins'])
    bin_mapping = jnp.array(bin_dict['bin_mapping'])
    surface_density = jnp.array(bin_dict['surface_density'])
    V_data = jnp.array(bin_dict['V_mean'])
    sigma_data = jnp.array(bin_dict['V_sigma'])
    h1_data = jnp.array(bin_dict['h1'])
    h2_data = jnp.array(bin_dict['h2'])
    h3_data = jnp.array(bin_dict['h3'])
    h4_data = jnp.array(bin_dict['h4'])
    v0 = jnp.array(bin_dict['v0'])
    s = jnp.array(bin_dict['s'])
    alpha, beta, gamma = bin_dict['orientation']

    # V_data_err = jnp.where(0.1 * jnp.fabs(V_data) < 10, 10, 0.1 * V_data)
    # sigma_data_err = jnp.where(0.1 * jnp.fabs(sigma_data) < 5, 5, 0.1 * sigma_data)
    # h1_data_err = jnp.where(0.1 * jnp.fabs(h1_data) < 0.03, 0.03, 0.1 * jnp.fabs(h1_data))
    # h2_data_err = jnp.where(0.1 * jnp.fabs(h2_data) < 0.03, 0.03, 0.1 * jnp.fabs(h2_data))
    # h3_data_err = jnp.where(0.1 * jnp.fabs(h3_data) < 0.03, 0.03, 0.1 * jnp.fabs(h3_data))
    # h4_data_err = jnp.where(0.1 * jnp.fabs(h4_data) < 0.03, 0.03, 0.1 * jnp.fabs(h4_data))
    V_data_err = jnp.array(bin_dict['V_mean_err'])
    sigma_data_err = jnp.array(bin_dict['V_sigma_err'])
    h1_data_err = jnp.array(bin_dict['h1_err'])
    h2_data_err = jnp.array(bin_dict['h2_err'])
    h3_data_err = jnp.array(bin_dict['h3_err'])
    h4_data_err = jnp.array(bin_dict['h4_err'])

    # df_Rzphi_data = pd.read_csv(path + 'mock_axisymmetric_disc_Rzphi.csv')
    # Rzphi_density_data = jnp.array(df_Rzphi_data['mass'].to_numpy()).astype(jnp.float32)
    with open(path + 'mock_axisymmetric_disc_Rzphi.pkl', 'rb') as f:
        Rzphi_density_data = pickle.load(f)

    R_grid, z_grid, phi_grid = Rzphi_density_data['R_grid'], Rzphi_density_data['z_grid'], Rzphi_density_data['phi_grid']
    dR = np.unique(R_grid)[1] - np.unique(R_grid)[0]
    dz = np.unique(z_grid)[1] - np.unique(z_grid)[0]
    dphi = np.unique(phi_grid)[1] - np.unique(phi_grid)[0]
    sample_for_integration = Rzphi_density_data['sample_for_integration']

    from scipy.stats import qmc
    X_regular_grid, Y_regular_grid = bin_dict['X_regular_grid'], bin_dict['Y_regular_grid']
    dX = jnp.unique(X_regular_grid)[1] - jnp.unique(X_regular_grid)[0]
    dY = jnp.unique(Y_regular_grid)[1] - jnp.unique(Y_regular_grid)[0]
    sampler = qmc.Sobol(d=3, scramble=False)
    sample = sampler.random_base2(m=10)


    dict_data = {
        # 'w0': w0,
        'v0': v0,
        's': s,

        # 'Rzphi_density_data': Rzphi_density_data,
        'XY_density_data': surface_density,
        'V_data': V_data,
        'V_data_err': V_data_err,
        'sigma_data': sigma_data,
        'sigma_data_err': sigma_data_err,
        'h1_data': h1_data,
        'h1_data_err': h1_data_err,
        'h2_data': h2_data,
        'h2_data_err': h2_data_err,
        'h3_data': h3_data,
        'h3_data_err': h3_data_err,
        'h4_data': h4_data,
        'h4_data_err': h4_data_err,
        'num_per_bin': num_per_bin,
        'bin_mapping': bin_mapping,
        'total_bins': total_bins.item(),

        'R_grid': R_grid,
        'z_grid': z_grid,
        'phi_grid': phi_grid,
        'dR': dR,
        'dz': dz,
        'dphi': dphi,
        'sample_for_integration': sample_for_integration,

        'X_regular_grid': X_regular_grid,
        'Y_regular_grid': Y_regular_grid,
        'dX': dX,
        'dY': dY,
        'sample_for_integration_XY': sample,
    }

    return dict_data

In [2]:
# path = '/content/drive/MyDrive/SchwarMAX-MCMC/'
path = '/Users/hanyuan/Dropbox/python_script/SchwarMAX/'
dict_data = get_dict_data(path)

def log_prior(theta,):
    if (6 < theta[0] < 10) and (8 < theta[1] < 12) and (-1 < theta[2] < 2) and (-1 < theta[3] < 1) and (-1 < theta[4] < 1)\
    and (0 <= theta[5] < jnp.pi) and (0 <= theta[6] < jnp.pi/2) and (0 <= theta[7] < jnp.pi):
        return 0.0  # log(1) = 0 for uniform prior
    return -np.inf  # log(0) = -inf for out-of-bounds

def log_prob(theta,):
    # print(theta)
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf

    ll = logl_density(theta, dict_data, dict_data['total_bins'])

    return ll + lp

ndim = 8
nwalkers = 16  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = np.array([9.2, 10, 0.3, 0., 0., jnp.pi/4, jnp.pi/4, jnp.pi/4])
# initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)
np.random.seed(42)
initial_pos = p0 + np.random.uniform(-0.3, 0.3, (nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 500, progress=True)

samples = sampler.get_chain(discard=200, flat=True)


params_bestfit = np.percentile(samples, axis=0, q=50)
logl_val = logl_density(params_bestfit, dict_data, dict_data['total_bins'])
print('Best-fit logL projection', logl_val)#

dict_data['logl_density_max'] = logl_val

logrho0_best_fit, logM_bulge_best_fit, \
logRd_disc_best_fit, logHs_disc_best_fit, logRs_bulge_best_fit, \
alpha_best_fit, beta_best_fit, gamma_best_fit = params_bestfit
# logMhalo_best_fit, logrho0_best_fit, logM_bulge_best_fit, logRh_disk_best_fit, logRs_disk_best_fit, logHs_disk_best_fit, logRs_bulge_best_fit,\
#       alpha_best_fit, beta_best_fit, gamma_best_fit, logLM_best_fit = (11.8, 8.8, 10.4, 1.2, 0.45, -0.24, -0.1, 30*np.pi/180, 20*np.pi/180, 0*np.pi/180, 0)

print('logrho0_best_fit',logrho0_best_fit)
print('logM_bulge_best_fit',logM_bulge_best_fit)
print('logRd_disc_best_fit',logRd_disc_best_fit)
print('logHs_disk_best_fit',logHs_disc_best_fit)
print('logRs_bulge_best_fit',logRs_bulge_best_fit)
print('alpha_best_fit',alpha_best_fit * 180 / np.pi)
print('beta_best_fit',beta_best_fit * 180 / np.pi)
print('gamma_best_fit',gamma_best_fit * 180 / np.pi)

params_halo_pot = {
    'logM': 11.8,
    'Rs':19,
    'a':1.0,
    'b':1.0,
    'c':1.0,
    'x_origin':0.0,
    'y_origin':0.0,
    'z_origin':0.0,
    'dirx':0.0,
    'diry':0.0,
    'dirz':1.0
}

params_disk_rho = {
    'rho0_disc': 10 ** logrho0_best_fit,
    'Rd_disc': 10 ** logRd_disc_best_fit,
    'hz_disc': 10 ** logHs_disc_best_fit,
    'x_origin': 0.0,
    'y_origin': 0.0,
    'z_origin': 0.0,
    'dirx': 0.0,
    'diry': 0.0,
    'dirz': 1.0,
    'alpha': alpha_best_fit * 180 / jnp.pi,
    'beta': beta_best_fit * 180 / jnp.pi,
    'gamma': gamma_best_fit * 180 / jnp.pi,
    'light_to_mass_ratio': 1,
    'logM_bulge': logM_bulge_best_fit,
    'Rs_bulge': 10 ** logRs_bulge_best_fit,
}

@jax.jit
def potential_func(x, y, z, dict_phi, params_halo):
    """ Returns Phi(R, z) """
    phi_halo = NFW_potential(x, y, z, params_halo)
    phi_disk = evaluate_phi_axisymmetric(x, y, z, dict_phi)
    return phi_halo + phi_disk

@jax.jit
def density_func(x, y, z, params):
    """ Returns Stellar Density nu(R, z) """
    # Double Exponential Disk
    val = DoubleExponentialDisk_density(x, y, z, params) + Dehnen_density(x, y, z, params)
    return val

bounds = jnp.array(
    [
        [-15.0, 15.0],  # x
        [-15.0, 15.0],  # y
        [-5.0, 5.0],    # z
    ],
    dtype=jnp.float32,
)

n_samples = 20_000
# n_x,n_y,n_z = 48, 48, 32

# key = jax.random.PRNGKey(0)
# sample_ic_dict = sample_from_density_grid(
#     key,
#     density_func,
#     params_disk_rho,
#     bounds,
#     n_samples=n_samples,
#     n_x=n_x,
#     n_y=n_y,
#     n_z=n_z,
# )
# samples = np.asarray(sample_ic_dict["samples"])
# dict_data['w0'] = samples

samples_x = np.random.normal(0.0, 5.0, size=(n_samples,))
samples_y = np.random.normal(0.0, 5.0, size=(n_samples,))
samples_z = np.random.normal(0.0, 2.5, size=(n_samples,))
w0 = jnp.array([samples_x, samples_y, samples_z]).T
dict_data['w0'] = w0

100%|██████████| 500/500 [02:30<00:00,  3.32it/s]

Best-fit logL projection -3.0566364365618086
logrho0_best_fit 8.769119135244418
logM_bulge_best_fit 9.778690775985314
logRd_disc_best_fit 0.4829816764926995
logHs_disk_best_fit -0.20681211575578326
logRs_bulge_best_fit 0.29597050976530537
alpha_best_fit 33.567716603809366
beta_best_fit 19.017932354499163
gamma_best_fit 89.17024102979586


In [3]:

# ground_truth = [
#     12.,
#     logrho0_best_fit,
#     logM_bulge_best_fit,
#     jnp.log10(19).item(),
#     logRd_disc_best_fit,
#     logHs_disc_best_fit,
#     logRs_bulge_best_fit+0.1,
#     alpha_best_fit,
#     beta_best_fit,
#     gamma_best_fit,
#     0.0,
#     1.61
# ]
# ground_truth = [
#     11.1875, 10.08161914 , 9.96619078 , 0.9975036 ,  0.76423168, -0.6755621,
#     0.76472051 , 0.40915259 , 0.15521095 , 2.15984495, -0.875      , 1.475
#     ]

ground_truth = [10.07, 9.22, 9.2, 1.95, 0.54, -0.94, 0.35, 0.73, 0.71, 2.45, -0.25, 1.48]

start = time.time()
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
logL.block_until_ready()  # Ensure computation finishes before timing
end = time.time()
print('time per logl evaluation', end - start, 's')
print(logL)

import time
start = time.time()
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
logL.block_until_ready()  # Ensure computation finishes before timing
end = time.time()
print('time per logl evaluation', end - start, 's')
print(logL)

time per logl evaluation 248.79024481773376 s
-2441.2829949205307
time per logl evaluation 243.74137210845947 s
-2441.2829949205307


In [ ]:
logL_ls = []
for i in tqdm(range (0, 30)):
  logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
  logL_ls.append(logL)
  print(logL)
plt.hist(logL_ls, bins = 50)

  3%|▎         | 1/30 [00:16<08:01, 16.60s/it]

-1030.3345405223981


  7%|▋         | 2/30 [00:33<07:45, 16.63s/it]

-1030.3345404339427


 10%|█         | 3/30 [00:49<07:28, 16.62s/it]

...

-1030.3345404802508


 30%|███       | 9/30 [02:29<05:49, 16.64s/it]

-1030.3345404124966


 33%|███▎      | 10/30 [02:46<05:32, 16.64s/it]

-1030.3345404546808


In [ ]:
jax.config.update("jax_log_compiles", True)

def log_prior(params):
    lp = 0
    for i in range (0, ndim):
        if (params[i]<=prior_uniform_low[i]) & (params[i]>=prior_uniform_high[i]):
            lp+= -jnp.inf
    return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

ground_truth = [
    11.5,
    logrho0_best_fit,
    logM_bulge_best_fit,
    jnp.log10(19).item(),
    logRd_disc_best_fit,
    logHs_disc_best_fit,
    logRs_bulge_best_fit,
    alpha_best_fit,
    beta_best_fit,
    gamma_best_fit,
    0.,
    1.6
]
prior_uniform_low =  [
    ground_truth[0] - 3,
    ground_truth[1] - 3,
    ground_truth[2] - 3,
    ground_truth[3]- 1,
    ground_truth[4]- 1,
    ground_truth[5]- 1,
    ground_truth[6]- 1,
    0,
    0,
    0,
    -2,
    0
]
prior_uniform_high = [
    ground_truth[0] + 3,
    ground_truth[1] + 3,
    ground_truth[2] + 3,
    ground_truth[3]+ 1,
    ground_truth[4]+ 1,
    ground_truth[5]+ 1,
    ground_truth[6]+ 1,
    jnp.pi,
    jnp.pi/2,
    jnp.pi,
    2,
    2
]

ndim = 12

n_grid = 1024
param_grid = pd.read_csv(path + '/quasi_random_samples_12D_unity.csv').to_numpy()
index = np.random.choice(len(param_grid), size=n_grid, replace=False)
param_grid = param_grid[index]
param_grid[:, 0] = (param_grid[:, 0] - 0.5) * 3 + ground_truth[0]
param_grid[:, 1] = (param_grid[:, 1] - 0.5) * 3 + ground_truth[1]
param_grid[:, 2] = (param_grid[:, 2] - 0.5) * 3 + ground_truth[2]
param_grid[:, 3] = (param_grid[:, 3] - 0.5) * 1.5 + ground_truth[3]
param_grid[:, 4] = (param_grid[:, 4] - 0.5) * 1.5 + ground_truth[4]
param_grid[:, 5] = (param_grid[:, 5] - 0.5) * 1.5 + ground_truth[5]
param_grid[:, 6] = (param_grid[:, 6] - 0.5) * 1.5 + ground_truth[6]
param_grid[:, 7] = (param_grid[:, 7] - 0.5) * 0.3 * jnp.pi + ground_truth[7]
param_grid[:, 8] = (param_grid[:, 8] - 0.5) * 0.3 * jnp.pi + ground_truth[8]
param_grid[:, 9] = (param_grid[:, 9]      ) * 1 * jnp.pi
param_grid[:, 10] = (param_grid[:, 10] - 0.5) * 2
param_grid[:, 11] = (param_grid[:, 11] - 0.5) * 0.4 + ground_truth[11]

from tqdm import tqdm
log_prob_grid = []
for i in tqdm(range(n_grid)):
  logl = log_prob(param_grid[i])
  log_prob_grid.append(logl)
  print(np.round(param_grid[i], 2), 'logL:', logl, "cache:", logl_angular_input._cache_size())

log_prob_grid = np.array(log_prob_grid)

pd.DataFrame({
    'logM_halo': param_grid[:, 0],
    'logM_disk': param_grid[:, 1],
    'logM_bulge': param_grid[:, 2],
    'logRs_halo': param_grid[:, 3],
    'logRs_disk': param_grid[:, 4],
    'logHs_disk': param_grid[:, 5],
    'logRs_bulge': param_grid[:, 6],
    'alpha': param_grid[:, 7],
    'beta': param_grid[:, 8],
    'gamma': param_grid[:, 9],
    'log_light_to_mass_ratio': param_grid[:, 10],
    'log_Omega': param_grid[:, 11],
    'log_prob': log_prob_grid,
}).to_csv(path + '/grid_search_result_0306.csv', index=False)

print("cache:", logl_angular_input._cache_size())

  0%|          | 1/1024 [01:25<24:13:05, 85.23s/it]

[12.29  8.27 11.27  1.35  0.21  0.53 -0.45  0.2   0.55  3.08  0.06  1.65] logL: -23829.248529295317 cache: 2


  0%|          | 2/1024 [01:50<14:08:55, 49.84s/it]

[10.82  8.47  9.95  0.83  0.92 -0.57 -0.02  0.97  0.47  1.67 -0.75  1.58] logL: -2897.678745630543 cache: 2
[11.21 10.19 10.42  1.5   0.27  0.03 -0.21  1.04  0.03  2.81 -0.62  1.58] logL: -inf cache: 2
[11.7   8.94 10.77  0.92  0.75 -0.52  0.59  1.05  0.45  1.31  0.4   1.64] logL: -inf cache: 2


  0%|          | 5/1024 [02:26<6:26:21, 22.75s/it] 

...

[12.23 10.25 10.8   2.    0.31 -0.8  -0.26  0.41  0.56  1.76 -0.19  1.79] logL: -6351.922786984564 cache: 2
[10.79  9.02  9.91  1.94  0.71  0.43 -0.28  0.58  0.2   1.78  0.98  1.8 ] logL: -inf cache: 2


100%|█████████▉| 1020/1024 [7:17:52<01:25, 21.43s/it]

[ 1.096e+01  8.330e+00  9.060e+00  6.600e-01  7.000e-01 -4.500e-01
 -3.700e-01  6.900e-01 -1.000e-02  2.010e+00 -2.200e-01  1.770e+00] logL: -2913.0543078734186 cache: 2


100%|██████████| 1024/1024 [7:18:28<00:00, 25.69s/it]

[10.15  7.32  9.03  1.2   0.21 -0.23  0.02  0.37  0.44  2.83  0.24  1.47] logL: -4805.933119129321 cache: 2
[11.46  8.26 11.05  1.87  0.76 -0.79  0.69  0.51  0.29  0.58  0.53  1.58] logL: -inf cache: 2
[11.39  9.4   8.54  1.44 -0.25  0.47  0.34  0.52  0.27  2.8   0.94  1.69] logL: -inf cache: 2
[10.99  9.48 11.19  0.68  0.26  0.2  -0.31  0.93  0.47  1.75  0.55  1.4 ] logL: -inf cache: 2
cache: 2


In [ ]:
from google.colab import runtime
runtime.unassign()